# Autoresearch workflow notebook

Run either backend from a notebook:

- **parametric**: existing `AutoresearchRunner` loop
- **mutation**: new `MutationRunner` sandboxed code-edit loop

You can select the backend with `AUTORESEARCH_NOTEBOOK_MODE`.


In [ ]:
%matplotlib inline

import os
from pathlib import Path
from pprint import pprint

from autoresearch import (
    AutoresearchRunner,
    BlackjackPolicyTrainer,
    BlackjackTask,
    HyperparameterTrainer,
    InventoryPolicyTrainer,
    LocalLLMResearchAgent,
    LocalLLMMutationAgent,
    MutationRunner,
    PROMPT_TEMPLATE_PRESETS,
    RestaurantInventoryTask,
    SafeExecutor,
    TinyTorchClassificationTask,
    TraceableAgent,
    TrainerRegistry,
    load_research_brief,
    plot_mutation_run,
    plot_run_result,
    save_html_report,
)
from autoresearch.demo import CyclicDemoAgent


In [ ]:
mode = os.getenv('AUTORESEARCH_NOTEBOOK_MODE', 'parametric').strip().lower()
TASK_NAME = os.getenv('AUTORESEARCH_NOTEBOOK_TASK', 'restaurant_inventory')
ITERATIONS = int(os.getenv('AUTORESEARCH_NOTEBOOK_ITERATIONS', '6'))
PROMPT_PRESET = os.getenv('AUTORESEARCH_PROMPT_PRESET', 'concise')
TEMPERATURE = float(os.getenv('AUTORESEARCH_TEMPERATURE', '0.2'))
BRIEF_PATH = os.getenv('AUTORESEARCH_BRIEF_PATH', 'research_brief.json')

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'autoresearch').exists() and (REPO_ROOT.parent / 'autoresearch').exists():
    REPO_ROOT = REPO_ROOT.parent
OUTPUT_ROOT = REPO_ROOT / 'artifacts' / 'notebooks'

registry = TrainerRegistry()
registry.register('restaurant_inventory', InventoryPolicyTrainer())
registry.register('blackjack', BlackjackPolicyTrainer())
registry.register('tiny_torch_classification', HyperparameterTrainer())

task = None
if mode == 'parametric':
    if TASK_NAME == 'restaurant_inventory':
        task = RestaurantInventoryTask()
    elif TASK_NAME == 'blackjack':
        task = BlackjackTask()
    elif TASK_NAME == 'tiny_torch_classification':
        task = TinyTorchClassificationTask()
    else:
        raise ValueError(f'Unsupported task: {TASK_NAME}')

print({
    'mode': mode,
    'task': TASK_NAME,
    'iterations': ITERATIONS,
    'prompt_preset': PROMPT_PRESET,
    'brief_path': BRIEF_PATH,
    'repo_root': str(REPO_ROOT),
})


In [ ]:
endpoint = os.getenv('AUTORESEARCH_AGENT_ENDPOINT', '').strip()
model = os.getenv('AUTORESEARCH_AGENT_MODEL', '').strip()

if mode == 'parametric':
    if endpoint and model:
        base_agent = LocalLLMResearchAgent.from_preset(
            endpoint=endpoint,
            model=model,
            prompt_preset=PROMPT_PRESET if PROMPT_PRESET in PROMPT_TEMPLATE_PRESETS else 'concise',
            temperature=TEMPERATURE,
        )
        agent_mode = 'local_llm'
    else:
        base_agent = CyclicDemoAgent()
        agent_mode = 'cyclic_demo'
    agent = TraceableAgent(base_agent)
else:
    if not (endpoint and model):
        raise ValueError('Mutation mode requires AUTORESEARCH_AGENT_ENDPOINT and AUTORESEARCH_AGENT_MODEL')
    agent = LocalLLMMutationAgent(
        endpoint=endpoint,
        model=model,
        temperature=TEMPERATURE,
    )
    agent_mode = 'local_mutation_llm'

print({'agent_mode': agent_mode, 'endpoint': endpoint or None, 'model': model or None})


In [ ]:
brief = None
brief_file = REPO_ROOT / BRIEF_PATH
if brief_file.exists():
    brief = load_research_brief(brief_file)

if mode == 'parametric':
    runner = AutoresearchRunner(
        agent=agent,
        registry=registry,
        research_brief=brief.to_context() if brief else None,
    )
    result = runner.run(task, iterations=ITERATIONS)
    print(f'Best score: {result.best_score:.6f}')
    pprint(result.best_model_state)
else:
    if brief is None:
        raise ValueError(f'Mutation mode requires a research brief file: {brief_file}')
    runner = MutationRunner(
        agent=agent,
        executor=SafeExecutor(
            timeout_seconds=int(brief.constraints.get('timeout_seconds', 120)),
            cpu_time_limit_seconds=brief.constraints.get('cpu_time_limit_seconds'),
            memory_limit_mb=brief.constraints.get('memory_limit_mb'),
        ),
    )
    result = runner.run(
        task_name=TASK_NAME,
        source_root=REPO_ROOT,
        research_brief=brief,
        iterations=ITERATIONS,
    )
    print(f'Mutation best score: {result.best_score:.6f}')
    print({'best_snapshot_id': result.best_snapshot_id, 'experiments': len(result.history)})


In [ ]:
if mode == 'parametric':
    fig = plot_run_result(result, show=False)
else:
    fig = plot_mutation_run(result, show=False)
fig


In [ ]:
if mode == 'parametric':
    history_rows = [
        {
            'iteration': entry.iteration,
            'score': entry.score,
            'suggestion': entry.suggestion,
            'model_state': entry.model_state,
            'metrics': entry.metrics,
        }
        for entry in result.history
    ]
else:
    history_rows = [
        {
            'iteration': entry.iteration,
            'status': entry.status.value,
            'description': entry.description,
            'score': entry.score,
            'failure_category': entry.failure_category,
            'snapshot_id': entry.snapshot_id,
            'auto_fixed': entry.auto_fixed,
            'resource_metrics': entry.resource_metrics,
        }
        for entry in result.history
    ]
history_rows


In [ ]:
output_dir = OUTPUT_ROOT / TASK_NAME
output_dir.mkdir(parents=True, exist_ok=True)
if mode == 'parametric':
    result.to_csv(output_dir / 'results.csv')
    result.to_trace_log(output_dir / 'trace.json')
    save_html_report(result, output_dir / 'report.html')
else:
    result.to_csv(output_dir / 'mutation_results.csv')
    result.to_experiment_log(output_dir / 'mutation_experiments.json')
print(f'Artifacts written to: {output_dir.resolve()}')
